# ATV04 — Importância das Variáveis por Remoção (LOFO) · versão **Pinguins de Palmer**

**Disciplina:** Aprendizado de Máquina Explicável — Prof. Dr. Anderson Ara (DEST/LEG/UFPR)
**Grupo:** Débora Zanoello · Marcelo Carvalho Rezende Faria · Allysson Souza · Alex St Clair

Mesmo problema usado em aula no Slide 10 (PDP): prever **P(sexo = fêmea)** a partir das medidas corporais, com **Random Forest** e **regressão logística**. Este notebook reproduz todos os números e figuras do deck *LOFO — Importância por Remoção (Pinguins)*.

| # | Teste | Pergunta | Slide |
|---|---|---|---|
| 1 | LOFO na Random Forest (CV estratificada 5×10) + teste de Nadeau & Bengio | Que variáveis trazem informação **única**? | 8 |
| 2 | PFI × LOFO no mesmo modelo | Remover e permutar dão a mesma resposta? | 9 |
| 3 | LOFO na regressão logística | A importância depende do algoritmo? | 10 |
| 4 | LOFO por grupos + diagnóstico de redundância | Variáveis substitutas “se cobrem”? | 11 |

**Definição.** Para uma perda $L$ avaliada fora do treino,

$$\mathrm{LOFO}_j = \mathbb{E}\big[L(Y, \hat f_{-j}(X_{-j}))\big] - \mathbb{E}\big[L(Y, \hat f(X))\big],$$

em que $\hat f_{-j}$ é o **mesmo algoritmo re-treinado sem** $X_j$. Aqui $L$ = **log-loss** (perda logarítmica), uma regra de pontuação própria. A acurácia aparece só como apoio à intuição.

## 0 · Ambiente

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sklearn

try:
    from palmerpenguins import load_penguins
except ImportError:                                  # pip install palmerpenguins
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "palmerpenguins"])
    from palmerpenguins import load_penguins

warnings.filterwarnings("ignore")
SEMENTE = 42
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

COR = {"sup": "#F7F6F1", "tinta": "#1B2233", "azul": "#2A68C2", "lar": "#D9572B",
       "amar": "#E0A400", "grade": "#E2E0D8"}
mpl.rcParams.update({"figure.facecolor": COR["sup"], "axes.facecolor": COR["sup"],
    "axes.grid": True, "axes.axisbelow": True, "grid.color": COR["grade"],
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "font.size": 11, "legend.frameon": False})

## 1 · Dados

`palmerpenguins` (Horst, Hill & Gorman, 2020; Gorman, Williams & Fraser, 2014). Removemos os registros sem sexo ou sem medidas.

In [ ]:
d = load_penguins().dropna(subset=["sex", "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"])
d = d.reset_index(drop=True)
NUM = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CAT = ["species", "island"]
FEATS = NUM + CAT
X_all, y_all = d[FEATS], (d["sex"] == "female").astype(int)
print(f"n = {len(d)} | fêmeas = {y_all.mean():.1%}")
display(d[FEATS + ["sex"]].head())
print(d[NUM].corr().round(2))

## 2 · Modelos e protocolo

- **Random Forest** (300 árvores, `min_samples_leaf=3`, `max_features=0.5`) e **regressão logística** (contínuas padronizadas, categóricas em one-hot dentro do pipeline).
- **CV estratificada 5 dobras × 10 repetições** — as mesmas 50 dobras para todos os ajustes, então as diferenças são pareadas.
- As categóricas são removidas **inteiras** (todas as dummies de uma vez), porque o pré-processamento é refeito para cada subconjunto de variáveis.

In [ ]:
def preproc(feats):
    num = [f for f in feats if f in NUM]; cat = [f for f in feats if f in CAT]
    tr = []
    if num: tr.append(("z", StandardScaler(), num))
    if cat: tr.append(("oh", OneHotEncoder(handle_unknown="ignore"), cat))
    return ColumnTransformer(tr)

def rf(feats):
    return make_pipeline(preproc(feats), RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                         max_features=0.5, random_state=SEMENTE, n_jobs=-1))

def logit(feats):
    return make_pipeline(preproc(feats), LogisticRegression(max_iter=2000))

DOBRAS = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMENTE).split(X_all, y_all))
GRUPOS = {f: [f] for f in FEATS}
GRUPOS.update({"{species,island}": ["species", "island"],
               "{bico}": ["bill_length_mm", "bill_depth_mm"],
               "{4 medidas}": NUM})
print("Dobras:", len(DOBRAS), "| variáveis e grupos avaliados:", len(GRUPOS))

## 3 · Implementação do LOFO e do teste de significância

Para cada dobra: treinar com todas as variáveis, medir a log-loss; para cada variável/grupo, **re-treinar sem ela** e medir de novo; $\Delta_{jk} = L_k^{(-j)} - L_k$.

**Teste t corrigido de Nadeau & Bengio (2003)**, porque as dobras de CV repetida não são independentes:
$t = \bar\Delta_j \,/\, \sqrt{(1/K + 1/4)\, s^2_{\Delta_j}}$, com $K-1$ graus de liberdade.

In [ ]:
def teste_nb(dd, razao=0.25):
    dd = np.asarray(dd); k = len(dd)
    t = dd.mean() / np.sqrt((1 / k + razao) * dd.var(ddof=1))
    return t, 2 * stats.t.sf(abs(t), k - 1)


def lofo(fab):
    base_ll, base_ac = [], []
    sem_ll, sem_ac = {g: [] for g in GRUPOS}, {g: [] for g in GRUPOS}
    for tr, te in DOBRAS:
        Xa, Xb, ya, yb = X_all.iloc[tr], X_all.iloc[te], y_all.iloc[tr], y_all.iloc[te]
        p = fab(FEATS).fit(Xa, ya).predict_proba(Xb)[:, 1]
        base_ll.append(log_loss(yb, p, labels=[0, 1])); base_ac.append(accuracy_score(yb, p > 0.5))
        for g, cols in GRUPOS.items():
            fs = [f for f in FEATS if f not in cols]
            p = fab(fs).fit(Xa[fs], ya).predict_proba(Xb[fs])[:, 1]
            sem_ll[g].append(log_loss(yb, p, labels=[0, 1])); sem_ac[g].append(accuracy_score(yb, p > 0.5))
    base_ll, base_ac = np.array(base_ll), np.array(base_ac)
    linhas, deltas = [], {}
    for g in GRUPOS:
        dd = np.array(sem_ll[g]) - base_ll; deltas[g] = dd
        t, p = teste_nb(dd)
        linhas.append({"variavel": g, "lofo_logloss": dd.mean(), "dp": dd.std(ddof=1),
                       "dobras_pos": (dd > 0).mean(), "t": t, "p_NB": p,
                       "queda_acuracia_pp": 100 * (base_ac - np.array(sem_ac[g])).mean()})
    tab = pd.DataFrame(linhas).sort_values("lofo_logloss", ascending=False).reset_index(drop=True)
    return tab, base_ll, base_ac, deltas

ROT = {"bill_length_mm": "compr. do bico", "bill_depth_mm": "prof. do bico", "flipper_length_mm": "nadadeira",
       "body_mass_g": "massa corporal", "species": "espécie", "island": "ilha",
       "{species,island}": "grupo {espécie, ilha}", "{bico}": "grupo {bico}", "{4 medidas}": "grupo {4 medidas}"}

## Teste 1 · LOFO na Random Forest

In [ ]:
t0 = time.time()
tab_rf, ll_rf, ac_rf, d_rf = lofo(rf)
print(f"Random Forest — log-loss {ll_rf.mean():.3f} | acurácia {ac_rf.mean():.1%} | "
      f"log-loss do acaso {log_loss(y_all, np.full(len(y_all), y_all.mean())):.3f} ({time.time()-t0:.0f} s)")
ind_rf = tab_rf[tab_rf["variavel"].isin(FEATS)].reset_index(drop=True)
display(ind_rf.round({"lofo_logloss": 4, "dp": 4, "dobras_pos": 2, "t": 2, "p_NB": 4, "queda_acuracia_pp": 1}))

In [ ]:
t = ind_rf.sort_values("lofo_logloss")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.1, 1]})
ax = axes[0]
ax.barh([ROT[v] for v in t["variavel"]], t["lofo_logloss"], xerr=1.96 * t["dp"] / np.sqrt(len(DOBRAS)),
        color=[COR["lar"] if v > 0 else COR["azul"] for v in t["lofo_logloss"]], height=0.6, error_kw={"capsize": 3})
ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
ax.set_xlabel("aumento da log-loss ao remover (IC 95%)"); ax.set_title("Teste 1 · LOFO — Random Forest")
ax = axes[1]
ordem = t["variavel"].tolist()
ax.boxplot([d_rf[v] for v in ordem], vert=False, widths=0.55)
ax.set_yticks(range(1, len(ordem) + 1), [ROT[v] for v in ordem]); ax.axvline(0, color=COR["tinta"], lw=1)
ax.grid(axis="y", visible=False); ax.set_xlabel("Δ log-loss por dobra (50 dobras)"); ax.set_title("Dobra a dobra")
plt.tight_layout(); plt.show()

**Leitura.** Nenhuma medida é indispensável sozinha. A maior perda é a da massa corporal (≈ +0,07 na log-loss, ≈ 4 pontos de acurácia). Bico e espécie são pequenos, mas significativos. A nadadeira é negativa (a floresta melhora sem ela na maioria das dobras) e a ilha não acrescenta nada.

## Teste 2 · PFI × LOFO no mesmo modelo

Importância por permutação (**PFI**) da Random Forest no hold-out estratificado de 30%, 50 permutações, com as variáveis **originais** (a espécie é permutada inteira).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.3, random_state=SEMENTE, stratify=y_all)
pfi = {}
for nome, fab in [("RF", rf), ("logística", logit)]:
    m = fab(FEATS).fit(X_tr, y_tr)
    r = permutation_importance(m, X_te, y_te, scoring="neg_log_loss", n_repeats=50, random_state=SEMENTE)
    pfi[nome] = pd.Series(r.importances_mean, index=FEATS)
comp2 = pd.DataFrame({"PFI (RF)": pfi["RF"], "LOFO (RF)": ind_rf.set_index("variavel")["lofo_logloss"]})
comp2["PFI / LOFO"] = comp2["PFI (RF)"] / comp2["LOFO (RF)"]
display(comp2.sort_values("LOFO (RF)", ascending=False).round(3))

In [ ]:
ordem = ind_rf.sort_values("lofo_logloss")["variavel"].tolist()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
axes[0].barh([ROT[v] for v in ordem], pfi["RF"][ordem], color=COR["azul"], height=0.6)
axes[0].set_title("PFI: permutar no modelo treinado"); axes[0].set_xlabel("Δ log-loss ao permutar")
v = ind_rf.set_index("variavel").loc[ordem, "lofo_logloss"]
axes[1].barh([ROT[x] for x in ordem], v, color=[COR["lar"] if x > 0 else COR["azul"] for x in v], height=0.6)
axes[1].set_title("LOFO: remover e re-treinar"); axes[1].set_xlabel("Δ log-loss ao remover")
for ax in axes:
    ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Leitura.** A ordem é parecida, mas a escala não: a PFI da massa é ≈ 4× o LOFO, e a da profundidade do bico, ≈ 8×. Ao permutar a massa, um Adélie pequeno pode receber o peso de um Gentoo — combinação que não existe nos dados, e o modelo **extrapola** (Hooker, Mentch & Zhou, 2021). No LOFO, o modelo re-treinado redistribui a informação entre as medidas.

## Teste 3 · Dependência do algoritmo: regressão logística

In [ ]:
t0 = time.time()
tab_lg, ll_lg, ac_lg, _ = lofo(logit)
print(f"Logística — log-loss {ll_lg.mean():.3f} | acurácia {ac_lg.mean():.1%} ({time.time()-t0:.0f} s)")
comp3 = (tab_rf.set_index("variavel")[["lofo_logloss", "p_NB"]]
         .join(tab_lg.set_index("variavel")[["lofo_logloss", "p_NB"]], lsuffix="_RF", rsuffix="_logística"))
display(comp3.loc[FEATS].sort_values("lofo_logloss_RF", ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, tb, tt in [(axes[0], tab_rf, "Random Forest"), (axes[1], tab_lg, "Regressão logística")]:
    v = tb.set_index("variavel").loc[ordem, "lofo_logloss"]
    ax.barh([ROT[x] for x in ordem], v, color=[COR["lar"] if x > 0 else COR["azul"] for x in v], height=0.6)
    ax.axvline(0, color=COR["tinta"], lw=1); ax.grid(axis="y", visible=False)
    ax.set_title(f"Teste 3 · LOFO — {tt}"); ax.set_xlabel("Δ log-loss")
plt.tight_layout(); plt.show()

**Leitura.** Os dois modelos acertam ≈ 91%, mas usam as variáveis de outro jeito: na logística a profundidade do bico passa a massa e a espécie pesa o dobro. Uma explicação plausível é que a logística, aditiva, precisa da espécie para ajustar o nível de cada grupo, enquanto a floresta reconstrói a espécie a partir das medidas. O LOFO explica o **par algoritmo + dados** (efeito Rashomon; Fisher, Rudin & Dominici, 2019).

## Teste 4 · LOFO por grupos e diagnóstico de redundância

Comparamos a soma dos LOFO individuais com o LOFO do grupo e verificamos **por que** espécie e ilha valem tão pouco.

In [ ]:
s = tab_rf.set_index("variavel")["lofo_logloss"]
qa = tab_rf.set_index("variavel")["queda_acuracia_pp"]
tab4 = pd.DataFrame({
    "soma dos individuais": [s[NUM].sum(), s[["bill_length_mm", "bill_depth_mm"]].sum(), s[["species", "island"]].sum()],
    "LOFO do grupo": [s["{4 medidas}"], s["{bico}"], s["{species,island}"]],
    "queda de acurácia do grupo (pp)": [qa["{4 medidas}"], qa["{bico}"], qa["{species,island}"]],
}, index=["4 medidas", "bico (compr. + prof.)", "espécie + ilha"])
display(tab4.round(3))
print(f"Acurácia sem as 4 medidas: {ac_rf.mean() - qa['{4 medidas}']/100:.1%}")

In [ ]:
acc_sp = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                         d[NUM], d["species"], cv=5).mean()
print(f"As 4 medidas preveem a ESPÉCIE com {acc_sp:.1%} de acerto (CV 5 dobras).")
print("\nEspécie × ilha:")
display(pd.crosstab(d["species"], d["island"]))

In [ ]:
gs = [("body_mass_g", "massa"), ("bill_length_mm", "compr.\nbico"), ("bill_depth_mm", "prof.\nbico"),
      ("{bico}", "grupo\n{bico}"), ("{4 medidas}", "grupo\n{4 medidas}"), ("species", "espécie"),
      ("island", "ilha"), ("{species,island}", "grupo\n{espécie, ilha}")]
vals = [s[g] for g, _ in gs]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(range(len(gs)), vals, color=[COR["amar"] if g.startswith("{") else COR["lar"] for g, _ in gs], width=0.6)
for i, (g, _) in enumerate(gs):
    ax.text(i, max(s[g], 0) + 0.01, f"{s[g]:.3f}\n({qa[g]:.0f} pp)", ha="center", fontsize=9)
ax.set_xticks(range(len(gs)), [n for _, n in gs]); ax.axhline(0, color=COR["tinta"], lw=1)
ax.set_ylabel("Δ log-loss ao remover"); ax.grid(axis="x", visible=False); ax.set_ylim(-0.02, 0.56)
ax.set_title("Teste 4 · Sozinhas valem pouco; juntas, valem tudo (queda de acurácia entre parênteses)")
plt.show()

**Leitura.** A soma dos LOFO individuais das 4 medidas é ≈ 0,13; retirando as quatro juntas, ≈ 0,46, e a acurácia cai para o nível do acaso. **As medidas se substituem.** A ilha vale ≈ 0 porque Chinstrap só vive em Dream e Gentoo só em Biscoe: a espécie já diz a ilha — e as medidas já dizem a espécie (≈ 99%). Regra prática: **definir os grupos antes de interpretar**.

## Resumo

| Teste | Resultado |
|---|---|
| 1 · LOFO RF | maior perda: massa (≈ +0,07; ≈ 4 pp); nadadeira negativa; ilha ≈ 0 |
| 2 · PFI × LOFO | mesma ordem, PFI 4–8× maior nas medidas (extrapolação) |
| 3 · Logística | desempenho igual, outra ordem (prof. do bico 1º; espécie ×2) |
| 4 · Grupos | 4 medidas: 0,13 somando × 0,46 em grupo; acurácia 91% → ≈ 42% |

### Referências
Lei et al. (2018), *JASA* 113(523) · Williamson et al. (2023), *JASA* 118(543) · Hooker, Mentch & Zhou (2021), *Statistics and Computing* 31:82 · Covert, Lundberg & Lee (2021), *JMLR* 22(209) · Fisher, Rudin & Dominici (2019), *JMLR* 20(177) · Molnar (2025), *Interpretable Machine Learning*, 3ª ed. · Nadeau & Bengio (2003), *Machine Learning* 52 · Gorman, Williams & Fraser (2014), *PLoS ONE* 9(3): e90081 · Horst, Hill & Gorman (2020), palmerpenguins.